In [1]:
import pandas as pd
import tqdm
import bte

Notebook to merge UShER tree strain IDs with metadata, generate TSV files, and run MetaFitch to infer the date and country of internal nodes using Fitch's algorithm.

In [32]:
tree = bte.MATree("//Users/reem/Downloads/public-2023-06-30.all.masked.pb")

Finished 'from_pb' in 49.1005 seconds


In [33]:
nwk = tree.write_newick("/Users/reem/tree_2023.nwk")

In [34]:
all_nodes = list(tree.depth_first_expansion())
nodes = []
for node in all_nodes:
    nodes.append({
        "strain": node.id})
all_nodes = pd.DataFrame(nodes)
all_nodes.head()
   

,strain
0,node_1
1,CHN/YN-0306-466/2020|MT396241.1|2020-03-06
2,DP0803|LC571037.1|2020-02-17
3,node_2
4,England/LEED-2A8B52/2020|OA971832.1|2020-04-04


In [35]:
usher_metadata =  pd.read_csv("/Users/reem/Downloads/public-2023-06-30.metadata.tsv", sep="\t")
usher_metadata.head()

/var/folders/bt/jpy5j9ms7pb6p6hhqzvpjffw0000gn/T/ipykernel_28193/3551984879.py:1: DtypeWarning: Columns (1,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  usher_metadata =  pd.read_csv("/Users/reem/Downloads/public-2023-06-30.metadata.tsv", sep="\t")


,strain,genbank_accession,date,country,host,completeness,length,Nextstrain_clade,pangolin_lineage,Nextstrain_clade_usher,pango_lineage_usher
0,100002|LR824035.1|2020-03-05,LR824035.1,2020-03-05,Switzerland,NaN,NaN,29903.0,20A,B.1,20A,B.1
1,100003|LR824038.1|2020-03-04,LR824038.1,2020-03-04,Switzerland,NaN,NaN,29903.0,20B,B.1.1,20B,B.1.1
2,100004|LR824040.1|2020-03-06,LR824040.1,2020-03-06,Switzerland,NaN,NaN,29903.0,20A,B.1,20A,B.1
3,100005|LR824037.1|2020-03-06,LR824037.1,2020-03-06,Switzerland,NaN,NaN,29903.0,20A,B.1,20A,B.1
4,100006|LR824041.1|2020-03-06,LR824041.1,2020-03-06,Switzerland,NaN,NaN,29903.0,20A,B.1,20A,B.1


In [36]:
usher_metadata.shape

(7611110, 11)

In [37]:

all_nodes.head()

,strain
0,node_1
1,CHN/YN-0306-466/2020|MT396241.1|2020-03-06
2,DP0803|LC571037.1|2020-02-17
3,node_2
4,England/LEED-2A8B52/2020|OA971832.1|2020-04-04


In [38]:
len(all_nodes)

9197366

In [39]:
# Merge date and country form metadata with bte_nodes on the strain col
metafitch_meta = all_nodes.merge(usher_metadata[["strain","country"]], on="strain", how="left")
metafitch_meta.head()

,strain,country
0,node_1,NaN
1,CHN/YN-0306-466/2020|MT396241.1|2020-03-06,China
2,DP0803|LC571037.1|2020-02-17,NaN
3,node_2,NaN
4,England/LEED-2A8B52/2020|OA971832.1|2020-04-04,England


In [40]:
metafitch_meta["country"].isna().sum()

np.int64(1591781)

In [41]:
# Remove internal node entries
metafitch_meta = metafitch_meta[~metafitch_meta["strain"].str.startswith("node_")]
metafitch_meta["country"].isna().sum()

np.int64(5758)

In [42]:
# Add country info for strains that start with hCoV-19 (added from nextclade analysis)
mask = metafitch_meta["strain"].str.startswith("hCoV-19")
metafitch_meta.loc[mask, "country"] = metafitch_meta.loc[mask, "strain"].str.split("/").str[1]
metafitch_meta["country"].isna().sum()


np.int64(5759)

In [12]:
metafitch_meta["country"] = metafitch_meta["country"].fillna("").str.strip()
metafitch_meta["strain"] = metafitch_meta["strain"].str.strip()
metafitch_meta.head(20)

,strain,country
1,CHN/YN-0306-466/2020|MT396241.1|2020-03-06,China
2,DP0803|LC571037.1|2020-02-17,
4,England/LEED-2A8B52/2020|OA971832.1|2020-04-04,England
5,England/SHEF-C06CE/2020|2020-03-25,England
8,England/SHEF-C07F8/2020|2020-03-21,England
9,England/SHEF-C0145/2020|2020-03-25,England
12,England/SHEF-CA0D5/2020|2020-04-02,England
13,USA/TX-HHD-4392170/2020|OM181113.1|2020-06-23,USA
15,England/SHEF-C038B/2020|2020-03-27,England
16,England/SHEF-D05A0/2020|2020-03-24,England


In [43]:
metafitch_meta.to_csv("/Users/reem/2023_metafitch_countries.tsv", sep="\t", index=False)

In [44]:
# Now create date metadata
df_dates = all_nodes.merge(usher_metadata[["strain","date"]], on="strain", how="left")
df_dates.head()

,strain,date
0,node_1,NaN
1,CHN/YN-0306-466/2020|MT396241.1|2020-03-06,2020-03-06
2,DP0803|LC571037.1|2020-02-17,2020-02-17
3,node_2,NaN
4,England/LEED-2A8B52/2020|OA971832.1|2020-04-04,2020-04-04


In [45]:
# Extract year from date column
df_dates["date"] = df_dates["date"].str[:4]

In [46]:
df_dates.head()

,strain,date
0,node_1,NaN
1,CHN/YN-0306-466/2020|MT396241.1|2020-03-06,2020
2,DP0803|LC571037.1|2020-02-17,2020
3,node_2,NaN
4,England/LEED-2A8B52/2020|OA971832.1|2020-04-04,2020


In [47]:
df_dates = df_dates[~df_dates["strain"].astype(str).str.startswith("node_")]
len(df_dates)

7611343

In [48]:
df_dates["date"].isna().sum()

np.int64(594)

In [19]:
mask = df_dates["date"].isna()
df_dates.loc[mask, "date"] = df_dates.loc[mask, "strain"].astype(str).str.split("/").str[2]
df_dates["date"].isna().sum()

np.int64(1244)

In [49]:
mask = df_dates["strain"].str.startswith("hCoV-19")
metafitch_meta.loc[mask, "date"] = metafitch_meta.loc[mask, "strain"].str.split("|").str[2][:4]
metafitch_meta["date"].isna().sum()

np.int64(7611342)

In [50]:
df_dates.isna().sum()

strain      0
date      594
dtype: int64

In [ ]:
df_dates = pd.read_csv("/Users/reem/metafitch_dates.tsv", sep="\t")
df_dates.head()

In [51]:
df_dates.to_csv("/Users/reem/2023_metafitch_dates.tsv", sep="\t", index=False)

In [57]:
# Already ran metafitch in terminal, once with the date metadata and once with the country metadata.
# Now we will merge the output files together to get a final file with both date and country metadata for each node in the tree.

countries = pd.read_csv("/Users/reem/metafitch_country_output.tsv", sep="\t")
dates = pd.read_csv("/Users/reem/metafitch_date_output_strict.tsv", sep="\t")


/var/folders/bt/jpy5j9ms7pb6p6hhqzvpjffw0000gn/T/ipykernel_28193/3574196033.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  dates = pd.read_csv("/Users/reem/metafitch_date_output_strict.tsv", sep="\t")


In [9]:
nans = countries[countries["country"].isna()]
print(nans.tail(40))

                                                     strain country
21242213                                CVR118|OX665681.2|?     NaN
21242214                                 CVR75|OY618295.1|?     NaN
21242215                                 CVR71|OY618207.1|?     NaN
21242216                                 CVR65|OY621529.1|?     NaN
21242217                            CAMB-746BA|OX684610.1|?     NaN
21242218                                EDB002|OY378539.1|?     NaN
21242288                                 CVR80|OY618329.1|?     NaN
21242618                           NOTT-10DF9C|OX682770.1|?     NaN
21242619                           NOTT-10DF6F|OX683148.1|?     NaN
21243041                                           node_217     NaN
21243042                           NOTT-10E8BE|OX672348.1|?     NaN
21243043                           NOTT-10EA7C|OX609157.1|?     NaN
21243287        Japan/DP0190/2020|EPI_ISL_416581|2020-02-15     NaN
21243325        Japan/DP0481/2020|EPI_ISL_416605

In [58]:
# Merge the two files on the strain column
merged = pd.merge(countries, dates, on="strain", how="outer")
len(merged)


21244182

In [59]:
merged["date"] = pd.to_numeric(merged["date"], errors="coerce").astype("Int64")
merged.head()

,strain,country,date
0,186951/USA/2023|PX732842.1|2023,USA,2023
1,2019_nCoV_Muc_IMB1|LR824570.1|?,Germany,<NA>
2,2020/BIKEN/A50-18|LC603287.1|?,NaN,<NA>
3,2020/BIKEN/B-1|LC603286.1|?,Japan,<NA>
4,2020/BIKEN/B-2|LC644163.1|2020-05,Japan,2020


In [60]:
merged["date"] = merged["date"].fillna(0).astype(int)
merged.head()

,strain,country,date
0,186951/USA/2023|PX732842.1|2023,USA,2023
1,2019_nCoV_Muc_IMB1|LR824570.1|?,Germany,0
2,2020/BIKEN/A50-18|LC603287.1|?,NaN,0
3,2020/BIKEN/B-1|LC603286.1|?,Japan,0
4,2020/BIKEN/B-2|LC644163.1|2020-05,Japan,2020


In [61]:
merged.to_csv("/Users/reem/merged_metafitch_strict.tsv", sep="\t", index=False)

In [ ]:
# THE END!